<a href="https://colab.research.google.com/github/AHamamd150/KEK_Lectures/blob/main/notebooks/Lecture_4_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 4 — Large Language Models: From Attention to Deployment

**Learning objectives**

By the end of this notebook you will be able to:

1. Explain what self-attention is and implement it from scratch with NumPy.
2. Load a pretrained LLM from Hugging Face and run inference.
3. Use a `pipeline` for convenient text generation.
4. Apply a chat template to format multi-turn conversations correctly.
5. Build a stateful, multi-turn chatbot that remembers history.
6. Use LangChain to inject a system prompt and chain components together.

---
> **How to use this notebook:** Read every markdown cell before running the code cell below it. Every line of code has a comment explaining *what* it does and *why*.

---
## Part 1 — Self-Attention from Scratch

### What is attention?

The **attention mechanism** is the core innovation of the Transformer architecture (Vaswani et al., 2017).  
It allows every token in a sequence to **look at every other token** and decide how much to "attend" to it when building its own representation.

### The three projections: Q, K, V

Given an input matrix **X** (tokens × features), we project it into three new matrices:

| Matrix | Full name | Role |
|--------|-----------|------|
| **Q** | Query | "What am I looking for?" |
| **K** | Key | "What do I contain?" |
| **V** | Value | "What information do I pass on?" |

### The attention formula

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Step by step:
1. `Q @ K.T` → dot-product scores: how similar is each query to each key?
2. `softmax(scores)` → turn scores into probabilities (they sum to 1 per row).
3. `weights @ V` → take a weighted average of value vectors.

In [ ]:
import numpy as np  # NumPy gives us fast matrix operations without needing a GPU

# ── Input matrix ──────────────────────────────────────────────────────────────
# Each row is one token; each column is one feature (embedding dimension).
# Here we have 3 tokens and 2 features — tiny, but enough to see the math.
X = np.array([[1., 0.],   # Token 1  e.g. the word "The"
              [0., 1.],   # Token 2  e.g. the word "cat"
              [0., 2.]]) # Token 3  e.g. the word "sat"

# ── Weight matrices ───────────────────────────────────────────────────────────
np.random.seed(0)  # Fix the random seed so every run gives the same numbers

# W_Q, W_K, W_V are learned during training; here we just initialise them randomly.
# Shape is (features_in × features_out) = (2 × 2).
W_Q = np.random.randn(2, 2)  # Query weight matrix
W_K = np.random.randn(2, 2)  # Key   weight matrix
W_V = np.random.randn(2, 2)  # Value weight matrix

# ── Project inputs into Q, K, V spaces ───────────────────────────────────────
# Matrix multiplication: each token vector is transformed by the weight matrix.
# Result shape: (3 tokens × 2 features) for each.
Q = X @ W_Q   # Query vectors — "what each token is searching for"
K = X @ W_K   # Key   vectors — "what each token advertises about itself"
V = X @ W_V   # Value vectors — "what each token will contribute if selected"

# ── Compute raw attention scores ──────────────────────────────────────────────
# Q @ K.T compares every query against every key via dot product.
# Higher dot product = higher similarity = more attention.
# Result shape: (3 × 3) — one score for every (query token, key token) pair.
scores = Q @ K.T

# ── Softmax: convert scores → probabilities ───────────────────────────────────
# np.exp makes all values positive (e^x is always > 0).
weights = np.exp(scores)
# Divide each row by its sum so every row sums to exactly 1 (probability distribution).
# keepdims=True preserves the (3,1) shape so broadcasting works correctly.
weights /= weights.sum(axis=1, keepdims=True)

# ── Weighted sum of values ────────────────────────────────────────────────────
# Each output token = a blend of all value vectors, weighted by attention.
# Token i "borrows" information from other tokens proportional to how much it attends to them.
output = weights @ V   # Shape: (3 × 2) — one output vector per input token

print("Attention weights (each row sums to 1):")
print(np.round(weights, 3))
print("\nAttention output:")
print(np.round(output, 3))

### 🔍 Reading the output

- Each **row** of `weights` is a probability distribution over the 3 tokens.
- A high value at position `[i, j]` means token `i` attends strongly to token `j`.
- The `output` matrix replaces each token's raw embedding with a **context-aware** representation — it now "knows" what its neighbours contain.

> **Note:** Real transformers divide scores by √d_k before softmax to prevent vanishingly small gradients when the embedding dimension is large. We omit it here for clarity.

---
## Part 2 — Loading a Pretrained LLM with Hugging Face

### What is Hugging Face `transformers`?

The `transformers` library gives us access to thousands of pretrained models with a consistent API.  
We need three things to run a model:

| Object | What it does |
|--------|--------------|
| `AutoTokenizer` | Converts raw text → token IDs (and back) |
| `AutoModelForCausalLM` | The actual neural network that predicts the next token |
| `pipeline` | A convenience wrapper that combines both into one callable |

### Model choice: `Qwen2.5-1.5B-Instruct`

We use Qwen 2.5 1.5B because:
- **Small enough** to run on a single consumer GPU (or even CPU).
- **Instruction-tuned** — it has been fine-tuned to follow user instructions in a chat format.
- **Strong performance** for its size.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch  # PyTorch is the deep-learning backend that Hugging Face runs on top of

# Install LangChain packages (needed later in Part 6)
# The ! prefix runs a shell command from inside the notebook
!pip install -q langchain langchain-huggingface

# ── Reproducibility seeds ─────────────────────────────────────────────────────
# Setting seeds makes sampling deterministic — same prompt → same output.
torch.manual_seed(42)       # Seed for CPU operations
torch.cuda.manual_seed(42)  # Seed for GPU operations (no-op if no GPU present)

In [ ]:
# ── Model identifier ─────────────────────────────────────────────────────────
# This is the Hugging Face Hub path: "organisation/model-name"
# Hugging Face will download the weights automatically on first run.
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
# ── Tokenizer ─────────────────────────────────────────────────────────────────
# The tokenizer turns text like "Hello world" into [9707, 1879] (token IDs)
# and back again. Every model has its own vocabulary, so always load the
# tokenizer that matches the model.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# ── Model weights ─────────────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # Use 16-bit floats instead of 32-bit.
                                 # Halves VRAM usage with negligible quality loss.
                                 # (float32 = 4 bytes/param; float16 = 2 bytes/param)
    device_map="auto",          # Automatically place layers on GPU if available,
                                 # spill to CPU if VRAM runs out.
)

---
## Part 3 — The `pipeline` API

A `pipeline` wraps the tokenizer + model into a single callable object.  
You pass it raw text; it returns generated text. No manual tokenisation needed.

### Key parameters explained

| Parameter | Effect |
|-----------|--------|
| `return_full_text=False` | Return only the *new* tokens, not the prompt echoed back |
| `max_new_tokens=200` | Hard cap on how many tokens the model may generate |
| `temperature=0.3` | Controls randomness. 0 = deterministic, 1 = very random, >1 = chaotic |

### Other pipeline tasks

The same API supports many NLP tasks — just change the first argument:

```python
"summarization"          # Condense a long document
"translation_en_to_fr"   # Translate English → French
"text-classification"    # Label text (e.g. topic detection)
"sentiment-analysis"     # Positive / negative / neutral
"zero-shot-classification" # Classify without task-specific training
"question-answering"     # Extract answers from a context passage
"fill-mask"              # Predict a [MASK] token (BERT-style)
"token-classification"   # Named entity recognition, POS tagging
```

In [ ]:
generator = pipeline(
    "text-generation",       # Task type — we want the model to continue/generate text
    model=model,             # The model object we loaded above
    tokenizer=tokenizer,     # The matching tokenizer
    return_full_text=False,  # Only return newly generated tokens, not the prompt
    max_new_tokens=200,      # Stop after at most 200 new tokens (prevents infinite loops)
    temperature=0.3,         # Low temperature → focused, predictable output
                             # Raise to 0.7–1.0 for more creative / varied responses
    max_length=None          # No overall length cap (we use max_new_tokens instead)
)

---
## Part 4 — Chat Templates & Single-Turn Generation

### Why do we need a chat template?

Instruction-tuned models are trained on conversations formatted with **special tokens** that mark who is speaking.  
For Qwen the format looks roughly like:

```
<|im_start|>user
Write an email...<|im_end|>
<|im_start|>assistant
```

If you feed the model raw text without these tokens, it does not "know" it is in a chat context and will produce poor results.

`tokenizer.apply_chat_template()` handles this formatting automatically — you just pass a list of message dicts.

In [ ]:
# ── Define the conversation as a list of message dicts ────────────────────────
# Each dict has:
#   "role"    → who is speaking: "user", "assistant", or "system"
#   "content" → what they said
messages = [
    {"role": "user", "content": "Write an email to Ahmed apologizing for missing the next meeting as I have a conflicting meeting at the same time."}
]

# ── Apply the chat template ───────────────────────────────────────────────────
# tokenize=False       → return a formatted string, not token IDs yet
#                        (the pipeline will tokenise for us)
# add_generation_prompt=True → append the assistant turn-start token
#                              so the model knows it should reply next
input_ = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("Formatted prompt sent to the model:")
print(input_)

In [ ]:
# ── Generate and display the response ─────────────────────────────────────────
# generator() returns a list of dicts; we take the first (and only) result.
# Each dict has a "generated_text" key with the model's reply as a string.
output = generator(input_)[0]
print(output["generated_text"])

---
## Part 5 — `model.generate()`: Lower-Level Control

`pipeline` is convenient but hides the details. For more control you can call `model.generate()` directly.  
This requires you to tokenise the input yourself first.

### When to use `model.generate()` over `pipeline`

- You need to inspect or manipulate **logits** (raw output probabilities).
- You want to use advanced decoding strategies (beam search, contrastive search, etc.).
- You need access to **attention maps** or **hidden states**.

### Key `generate()` parameters

| Parameter | Effect |
|-----------|--------|
| `do_sample=True` | Sample from the distribution instead of always picking the top token |
| `temperature` | Scales logits before sampling (same meaning as in `pipeline`) |
| `max_new_tokens` | Maximum number of tokens to generate |

In [ ]:
# ── Tokenise the prompt manually ──────────────────────────────────────────────
# return_tensors="pt" → return PyTorch tensors (not Python lists)
# .to(model.device)   → move the tensor to the same device as the model (GPU/CPU)
inputs = tokenizer(input_, return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]  # Shape: (1, sequence_length)

# ── Generate ──────────────────────────────────────────────────────────────────
output_ids = model.generate(
    input_ids=input_ids,   # The tokenised prompt
    max_new_tokens=200,    # Cap the response length
    temperature=0.2,       # Very focused output (close to greedy)
    do_sample=True,        # Must be True when temperature != 1.0
)

# ── Decode the output back to text ────────────────────────────────────────────
# output_ids includes the prompt tokens too; we slice them off with [input_ids.shape[1]:]
new_tokens = output_ids[0][input_ids.shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

---
## Part 6 — Multi-Turn Chatbot with Conversation History

### The stateless problem

LLMs have **no memory between calls**. Each `generate()` call is independent.  
To simulate a conversation we must **manually pass the full history** on every turn.

### How it works

```
Turn 1:  history = [user: "Hi"]                       → model replies "Hello!"
Turn 2:  history = [user: "Hi", asst: "Hello!", user: "How are you?"]  → model replies
Turn 3:  history = [...all previous..., user: "..."]  → model replies
```

The history list grows with every exchange. The model can "remember" because we keep giving it the full context.

In [ ]:
history = []  # Starts empty; we append messages as the conversation grows

print("Chat started! Type 'exit' or 'quit' to end the session.")
print("-" * 50)

while True:
    # ── Get user input ────────────────────────────────────────────────────────
    user_input = input("You: ").strip()  # .strip() removes accidental leading/trailing spaces

    # ── Exit conditions ───────────────────────────────────────────────────────
    if user_input.lower() in ("exit", "quit"):
        print("Chat ended. Goodbye!")
        break  # Exit the while loop

    if not user_input:  # Ignore empty lines (user just pressed Enter)
        continue        # Jump back to the top of the loop

    # ── Append the new user message to history ────────────────────────────────
    history.append({"role": "user", "content": user_input})

    # ── Format the FULL conversation history as a prompt ─────────────────────
    # The model needs to see all previous turns to maintain coherence.
    prompt = tokenizer.apply_chat_template(
        history,                       # The entire conversation so far
        tokenize=False,                # Return a string, not token IDs
        add_generation_prompt=True     # Signal the model to generate the next assistant turn
    )

    # ── Generate the assistant's reply ───────────────────────────────────────
    output = generator(prompt)
    assistant_text = output[0]["generated_text"]  # Extract the string from the result dict

    # ── Save the assistant reply to history ───────────────────────────────────
    # IMPORTANT: we must store the reply so the model can refer to it next turn.
    history.append({"role": "assistant", "content": assistant_text})

    print(f"Your LLM: {assistant_text}")
    print("-" * 50)

---
## Part 7 — LangChain: Prompt Engineering with a System Prompt

### Why LangChain?

LangChain provides a composable, framework-agnostic way to build LLM applications.  
Instead of manually formatting strings, you define **prompt templates** and **chain** them together.

### The three components here

```
ChatPromptTemplate  →  HuggingFacePipeline  →  response
        ↑                      ↑
  formats the prompt      wraps your local model
```

### System vs User messages

| Role | Purpose | Example |
|------|---------|---------|
| `system` | Sets the model's persona, style, and constraints. Processed first. | "You are a concise assistant who replies in bullet points." |
| `user` | The actual question or instruction from the end user. | "What are transformers?" |
| `assistant` | The model's reply (used in multi-turn history). | — |

> A **system prompt** is one of the most powerful prompt engineering tools. It lets you control tone, format, language, and behaviour without changing the user-facing prompt.

In [ ]:
from langchain_huggingface import HuggingFacePipeline  # Adapts a HF pipeline for LangChain
from langchain_core.prompts import ChatPromptTemplate   # Template with typed roles

In [ ]:
# ── Step 1: Wrap the pipeline ─────────────────────────────────────────────────
# LangChain needs to call the model through its own interface.
# HuggingFacePipeline adapts our existing `generator` object — no second model load.
llm = HuggingFacePipeline(pipeline=generator)

In [ ]:
# ── Step 2: Define the prompt template ────────────────────────────────────────
# from_messages() accepts a list of (role, content) tuples.
# {user_input} is a placeholder — it will be filled in at call time (see Step 4).
prompt = ChatPromptTemplate.from_messages([
    # System message: permanent instructions that shape the model's behaviour.
    # The model will follow these constraints for every user query in this chain.
    ("system", "You are a concise assistant who always replies in bullet points. "
               "Use math equations where relevant."),
    # Human message: the variable part — different every time we call the chain.
    ("human", "{user_input}"),
])

In [ ]:
# ── Step 3: Build the chain ───────────────────────────────────────────────────
# The pipe operator | connects components left to right:
#   1. `prompt` formats the input into a full conversation string.
#   2. `llm` receives that string and generates a reply.
# The result is a reusable Runnable — call it as many times as you like.
chain = prompt | llm

In [ ]:
# ── Step 4: Run the chain ─────────────────────────────────────────────────────
# invoke() fills in the {user_input} placeholder and runs the whole chain.
# The dict keys must match the placeholder names in the template.
response = chain.invoke({"user_input": "What are the key ideas behind transformer models?"})
print(response)

---
## Summary

| Concept | Key takeaway |
|---------|-------------|
| **Self-attention** | Each token attends to all others via Q·Kᵀ dot products, then blends V vectors. |
| **Tokeniser** | Converts text ↔ integer IDs; each model has its own vocabulary. |
| **`from_pretrained`** | Downloads weights from Hugging Face Hub; `torch_dtype=float16` halves memory. |
| **`pipeline`** | High-level wrapper — pass text in, get text out. |
| **Chat template** | Formats the conversation with the special tokens the model was trained on. |
| **History loop** | Append every turn to a list and pass the whole list each time. |
| **LangChain** | Composable prompt templates + model wrappers; system prompt controls behaviour. |
